In [1]:
import numpy as np                                          # NumPy for all array math (fields, grids, interpolation)
import matplotlib.pyplot as plt                             # Matplotlib for plotting and figure creation
from matplotlib.animation import FuncAnimation, PillowWriter # FuncAnimation builds frame-by-frame animations; PillowWriter saves them as GIF
from mpl_toolkits.mplot3d.art3d import Line3DCollection     # Efficient way to draw many 3D line segments at once (used for streamlines)

In [2]:
Nx, Ny, Nz = 90, 44, 44              # Number of interior grid cells in x (flow direction), y, and z
SX, SY, SZ = Nx + 2, Ny + 2, Nz + 2  # Total array size including 1 ghost/boundary cell on each side
R = 10.0                             # Radius of the sphere obstacle (in grid cells)
cx, cy, cz = 20.0, Ny / 2 + 1.0, Nz / 2 + 1.0  # Sphere center: x=20 (near inlet), centered in y and z (+1 for ghost cell offset)
U_in = 1.0                           # Inflow velocity at the inlet (x = 0)
dt = 0.4                             # Time step size
GS_ITERS = 26                        # Number of relaxation iterations for the diffusion and pressure solvers
Re = 2000.0                          # Reynolds number (ratio of inertial to viscous forces)
nu = U_in * (2 * R) / Re             # Kinematic viscosity from Re = U*D/nu, where D = 2R is sphere diameter -> nu = 0.01
CONFINEMENT_EPS = 0.5                # Strength of vorticity confinement (re-injects small swirls lost to numerical damping)
N_STEPS = 320                        # Total number of time steps to simulate
SNAPSHOT_EVERY = 3                   # Save the velocity field every 3 steps for the animation
FPS = 8                              # Frames per second of the output GIF

In [3]:
xx, yy, zz = np.meshgrid(np.arange(SX), np.arange(SY), np.arange(SZ), indexing='ij')  # 3D integer index arrays for every cell (ij = x,y,z order)
mask = (xx - cx) ** 2 + (yy - cy) ** 2 + (zz - cz) ** 2 <= R ** 2                     # Boolean array: True for cells inside the sphere (solid region)

In [4]:
def apply_vel_bc(u, v, w):                                   # Applies boundary conditions to all three velocity components
    u[0,:,:]=U_in; v[0,:,:]=0.0; w[0,:,:]=0.0                # Inlet (x=0): uniform flow U_in in x, no sideways velocity
    u[-1,:,:]=u[-2,:,:]; v[-1,:,:]=v[-2,:,:]; w[-1,:,:]=w[-2,:,:]  # Outlet (x=end): zero-gradient, flow leaves freely
    v[:,0,:]=0.0; v[:,-1,:]=0.0                              # y-walls: no flow through the wall (normal velocity v = 0)
    u[:,0,:]=u[:,1,:]; u[:,-1,:]=u[:,-2,:]                   # y-walls: u copies neighbor (free-slip, no friction)
    w[:,0,:]=w[:,1,:]; w[:,-1,:]=w[:,-2,:]                   # y-walls: w copies neighbor (free-slip)
    w[:,:,0]=0.0; w[:,:,-1]=0.0                              # z-walls: no flow through the wall (normal velocity w = 0)
    u[:,:,0]=u[:,:,1]; u[:,:,-1]=u[:,:,-2]                   # z-walls: u copies neighbor (free-slip)
    v[:,:,0]=v[:,:,1]; v[:,:,-1]=v[:,:,-2]                   # z-walls: v copies neighbor (free-slip)

def apply_scalar_bc(field, kind):                            # Same BCs as above but for ONE component, identified by 'u','v','w'
    field[0,:,:] = U_in if kind=='u' else 0.0                # Inlet: u = U_in, v and w = 0
    field[-1,:,:] = field[-2,:,:]                            # Outlet: zero-gradient for any component
    if kind=='v':                                            # If this is the y-velocity...
        field[:,0,:]=0.0; field[:,-1,:]=0.0                  # ...it must be zero on the y-walls (no penetration)
    else:                                                    # Otherwise (u or w)...
        field[:,0,:]=field[:,1,:]; field[:,-1,:]=field[:,-2,:]  # ...zero-gradient on y-walls (free-slip)
    if kind=='w':                                            # If this is the z-velocity...
        field[:,:,0]=0.0; field[:,:,-1]=0.0                  # ...it must be zero on the z-walls (no penetration)
    else:                                                    # Otherwise (u or v)...
        field[:,:,0]=field[:,:,1]; field[:,:,-1]=field[:,:,-2]  # ...zero-gradient on z-walls (free-slip)

def apply_pressure_bc(p):                                    # Boundary conditions for pressure (also reused for divergence)
    p[0,:,:]=p[1,:,:]; p[-1,:,:]=0.0                         # Inlet: zero-gradient (Neumann); Outlet: p = 0 (reference pressure, Dirichlet)
    p[:,0,:]=p[:,1,:]; p[:,-1,:]=p[:,-2,:]                   # y-walls: zero pressure gradient
    p[:,:,0]=p[:,:,1]; p[:,:,-1]=p[:,:,-2]                   # z-walls: zero pressure gradient

In [5]:
def diffuse_component(x, x0, diff_rate, dt, kind, iters=GS_ITERS):  # Implicitly diffuses one velocity component (viscous term)
    a = dt*diff_rate; c_inv = 1.0/(1+6*a)                            # a = nu*dt (grid spacing = 1); c_inv is the 1/(1+6a) denominator
    for _ in range(iters):                                           # Repeat the relaxation update to converge the implicit solve
        x[1:-1,1:-1,1:-1] = (x0[1:-1,1:-1,1:-1] + a*(                # New value = (old value + a * sum of 6 neighbors) / (1+6a)
            x[0:-2,1:-1,1:-1]+x[2:,1:-1,1:-1]+                       # Neighbors in x (left and right)
            x[1:-1,0:-2,1:-1]+x[1:-1,2:,1:-1]+                       # Neighbors in y (below and above)
            x[1:-1,1:-1,0:-2]+x[1:-1,1:-1,2:]))*c_inv                # Neighbors in z (back and front), then divide by (1+6a)
        apply_scalar_bc(x, kind)                                     # Re-enforce boundary conditions after each iteration
    return x                                                         # Return the diffused field

In [6]:
def advect(d, d0, u, v, w, dt, kind):                                         # Moves field d0 along the velocity field -> result in d
    Xg,Yg,Zg = np.meshgrid(np.arange(1,Nx+1), np.arange(1,Ny+1), np.arange(1,Nz+1), indexing='ij')  # Coordinates of all interior cells
    x = np.clip(Xg - dt*u[1:-1,1:-1,1:-1], 0.5, Nx+0.5)                       # Trace backward in x: where did this fluid come from? Clamp to domain
    y = np.clip(Yg - dt*v[1:-1,1:-1,1:-1], 0.5, Ny+0.5)                       # Trace backward in y, clamped
    z = np.clip(Zg - dt*w[1:-1,1:-1,1:-1], 0.5, Nz+0.5)                       # Trace backward in z, clamped
    i0=x.astype(int); i1=i0+1; j0=y.astype(int); j1=j0+1; k0=z.astype(int); k1=k0+1  # Indices of the 8 surrounding grid corners
    sx1=x-i0; sx0=1-sx1; sy1=y-j0; sy0=1-sy1; sz1=z-k0; sz0=1-sz1             # Fractional weights for trilinear interpolation in each axis
    d[1:-1,1:-1,1:-1] = (sx0*(sy0*(sz0*d0[i0,j0,k0]+sz1*d0[i0,j0,k1])+sy1*(sz0*d0[i0,j1,k0]+sz1*d0[i0,j1,k1]))+  # Blend the 4 corners on the i0 side
                          sx1*(sy0*(sz0*d0[i1,j0,k0]+sz1*d0[i1,j0,k1])+sy1*(sz0*d0[i1,j1,k0]+sz1*d0[i1,j1,k1])))  # ...plus the 4 corners on the i1 side
    apply_scalar_bc(d, kind)                                                  # Enforce boundary conditions on the advected field
    return d                                                                  # Return the advected field

In [7]:
def project(u, v, w, p, div):                                                 # Makes velocity divergence-free (incompressible)
    div[1:-1,1:-1,1:-1] = -(1.0/3.0)*0.5*((u[2:,1:-1,1:-1]-u[0:-2,1:-1,1:-1])+  # Central-difference du/dx, scaled (the 1/3 is a heuristic factor)...
                                            (v[1:-1,2:,1:-1]-v[1:-1,0:-2,1:-1])+  # ...plus dv/dy...
                                            (w[1:-1,1:-1,2:]-w[1:-1,1:-1,0:-2]))  # ...plus dw/dz -> negative divergence
    p[:] = 0                                                                  # Start the pressure solve from zero
    apply_pressure_bc(div); apply_pressure_bc(p)                              # Apply BCs to divergence and initial pressure
    for _ in range(GS_ITERS):                                                 # Iteratively solve the Poisson equation lap(p) = div
        p[1:-1,1:-1,1:-1] = (div[1:-1,1:-1,1:-1]+p[0:-2,1:-1,1:-1]+p[2:,1:-1,1:-1]+  # p = (div + x-neighbors...
                              p[1:-1,0:-2,1:-1]+p[1:-1,2:,1:-1]+p[1:-1,1:-1,0:-2]+p[1:-1,1:-1,2:])/6.0  # ...+ y and z neighbors) / 6
        apply_pressure_bc(p)                                                  # Re-apply pressure BCs each iteration
    u[1:-1,1:-1,1:-1] -= 0.5*(p[2:,1:-1,1:-1]-p[0:-2,1:-1,1:-1])              # Subtract dp/dx from u
    v[1:-1,1:-1,1:-1] -= 0.5*(p[1:-1,2:,1:-1]-p[1:-1,0:-2,1:-1])              # Subtract dp/dy from v
    w[1:-1,1:-1,1:-1] -= 0.5*(p[1:-1,1:-1,2:]-p[1:-1,1:-1,0:-2])              # Subtract dp/dz from w
    apply_vel_bc(u, v, w)                                                     # Re-apply velocity BCs after correction
    return u, v, w                                                            # Return the (approximately) divergence-free velocity

In [8]:
def vorticity_confinement(u, v, w, dt, eps):                                  # Adds a force that strengthens swirls (counters numerical damping)
    if eps <= 0:                                                              # If confinement is turned off...
        return u, v, w                                                        # ...return velocity unchanged
    dwdy = np.gradient(w, axis=1); dvdz = np.gradient(v, axis=2)              # Derivatives needed for omega_x
    dudz = np.gradient(u, axis=2); dwdx = np.gradient(w, axis=0)              # Derivatives needed for omega_y
    dvdx = np.gradient(v, axis=0); dudy = np.gradient(u, axis=1)              # Derivatives needed for omega_z
    omega_x = dwdy - dvdz; omega_y = dudz - dwdx; omega_z = dvdx - dudy       # Vorticity vector omega = curl(velocity)
    omega_mag = np.sqrt(omega_x**2+omega_y**2+omega_z**2) + 1e-8              # Vorticity magnitude (+tiny value avoids division by zero)
    gx = np.gradient(omega_mag, axis=0); gy = np.gradient(omega_mag, axis=1); gz = np.gradient(omega_mag, axis=2)  # Gradient of |omega|
    gnorm = np.sqrt(gx**2+gy**2+gz**2) + 1e-8                                 # Length of that gradient (+tiny value for safety)
    Nx_=gx/gnorm; Ny_=gy/gnorm; Nz_=gz/gnorm                                  # Unit vector N pointing toward stronger vorticity
    fx = eps*(Ny_*omega_z - Nz_*omega_y); fy = eps*(Nz_*omega_x - Nx_*omega_z); fz = eps*(Nx_*omega_y - Ny_*omega_x)  # Force f = eps * (N x omega)
    u += dt*fx; v += dt*fy; w += dt*fz                                        # Apply the force to the velocity over one time step
    return u, v, w                                                            # Return the updated velocity

In [9]:
u = np.full((SX,SY,SZ), U_in); v = np.zeros((SX,SY,SZ)); w = np.zeros((SX,SY,SZ))  # Start with uniform flow U_in in x, zero in y and z
u[mask] = 0.0                                                                # No flow inside the sphere
snapshots = []                                                               # List to hold saved velocity fields for animation
print("Solving...")                                                          # Progress message
for step in range(N_STEPS):                                                  # Main time-stepping loop
    u0,v0,w0 = u.copy(),v.copy(),w.copy()                                    # Save current velocity as the "old" state for diffusion
    diffuse_component(u,u0,nu,dt,'u'); diffuse_component(v,v0,nu,dt,'v'); diffuse_component(w,w0,nu,dt,'w')  # Viscous diffusion of each component
    p = np.zeros((SX,SY,SZ)); div = np.zeros((SX,SY,SZ))                     # Fresh arrays for pressure and divergence
    project(u,v,w,p,div); u[mask]=0.0; v[mask]=0.0; w[mask]=0.0              # Make flow incompressible, then zero velocity inside the sphere
    u0,v0,w0 = u.copy(),v.copy(),w.copy()                                    # Save velocity again as the source for advection
    advect(u,u0,u0,v0,w0,dt,'u'); advect(v,v0,u0,v0,w0,dt,'v'); advect(w,w0,u0,v0,w0,dt,'w')  # Velocity advects itself (nonlinear term)
    project(u,v,w,p,div); u[mask]=0.0; v[mask]=0.0; w[mask]=0.0              # Project again after advection, re-zero the sphere
    u,v,w = vorticity_confinement(u,v,w,dt,CONFINEMENT_EPS)                  # Add vorticity confinement force
    u[mask]=0.0; v[mask]=0.0; w[mask]=0.0                                    # Re-zero the sphere (confinement may have added velocity there)
    apply_vel_bc(u,v,w)                                                      # Final boundary condition enforcement for this step
    if step < 10:                                                            # For the first 10 steps only...
        v[int(cx):int(cx)+6, Ny//2:Ny//2+2, :] += 0.25                       # ...push a small y-kick behind the sphere to break symmetry
        w[int(cx):int(cx)+6, :, Nz//2:Nz//2+2] += 0.15                       # ...and a small z-kick, so the wake becomes unsteady
    if step % SNAPSHOT_EVERY == 0:                                           # Every SNAPSHOT_EVERY steps...
        snapshots.append((u.astype(np.float32).copy(), v.astype(np.float32).copy(), w.astype(np.float32).copy()))  # ...store a float32 copy (saves memory)
        if len(snapshots) % 20 == 0:                                         # Every 20 snapshots...
            print(f"  step {step}/{N_STEPS}  (snapshot {len(snapshots)})")   # ...print progress

print(f"Got {len(snapshots)} snapshots -> animation will be {len(snapshots)/FPS:.1f} seconds long")  # Report count and GIF duration (~107 frames, ~13 s)

Solving...
  step 57/320  (snapshot 20)
  step 117/320  (snapshot 40)
  step 177/320  (snapshot 60)
  step 237/320  (snapshot 80)
  step 297/320  (snapshot 100)
Got 107 snapshots -> animation will be 13.4 seconds long


In [10]:
# Streamline tracing

def sample_vel_vec(u, v, w, xs, ys, zs):                                     # Gets velocity at arbitrary (non-grid) points
    xs=np.clip(xs,0.5,Nx+0.5); ys=np.clip(ys,0.5,Ny+0.5); zs=np.clip(zs,0.5,Nz+0.5)  # Keep sample points inside the domain
    i0=xs.astype(int); i1=i0+1; j0=ys.astype(int); j1=j0+1; k0=zs.astype(int); k1=k0+1  # Indices of the 8 surrounding grid corners
    sx1=xs-i0; sx0=1-sx1; sy1=ys-j0; sy0=1-sy1; sz1=zs-k0; sz0=1-sz1         # Trilinear interpolation weights
    def interp(f):                                                           # Inner helper: trilinearly interpolate a single field f
        return (sx0*(sy0*(sz0*f[i0,j0,k0]+sz1*f[i0,j0,k1])+sy1*(sz0*f[i0,j1,k0]+sz1*f[i0,j1,k1]))+  # Weighted sum of i0-side corners
                sx1*(sy0*(sz0*f[i1,j0,k0]+sz1*f[i1,j0,k1])+sy1*(sz0*f[i1,j1,k0]+sz1*f[i1,j1,k1])))  # ...plus i1-side corners
    return interp(u), interp(v), interp(w)                                   # Return interpolated velocity components

def trace_streamlines(u, v, w, y0s, z0s, n_steps=160, ds=0.9, x_start=2.0):  # Traces streamlines starting near the inlet
    xs=np.full_like(y0s,x_start); ys=y0s.copy(); zs=z0s.copy()               # All seeds start at x = x_start with given y, z
    path=[np.stack([xs,ys,zs],axis=1)]                                       # Record starting positions (shape: n_seeds x 3)
    active=np.ones_like(xs,dtype=bool)                                       # Tracks which streamlines are still moving
    for _ in range(n_steps):                                                 # Integrate for n_steps steps
        vx,vy,vz = sample_vel_vec(u,v,w,xs,ys,zs)                            # Velocity at each current point
        speed=np.sqrt(vx**2+vy**2+vz**2)                                     # Speed at each point
        active &= (speed>1e-3)&(xs<Nx-0.5)&(xs>0.5)&(ys>0.5)&(ys<Ny-0.5)&(zs>0.5)&(zs<Nz-0.5)  # Stop lines that stall or leave the domain
        xs=xs+ds*np.where(active,vx,0.0); ys=ys+ds*np.where(active,vy,0.0); zs=zs+ds*np.where(active,vz,0.0)  # Forward Euler step (only active lines move)
        path.append(np.stack([xs,ys,zs],axis=1))                             # Record new positions
    return np.stack(path,axis=0)                                             # Return array of shape (n_steps+1, n_seeds, 3)

In [11]:
#Seed points and tracing all snapshots

y_seeds = np.linspace(6, Ny-6, 10)                                           # 10 evenly spaced seed positions in y (away from walls)
z_seeds = np.linspace(6, Nz-6, 8)                                            # 8 evenly spaced seed positions in z
Y0,Z0 = np.meshgrid(y_seeds, z_seeds)                                        # 2D grid of seed (y, z) pairs -> 80 seeds
y_flat, z_flat = Y0.flatten(), Z0.flatten()                                  # Flatten to 1D arrays for vectorized tracing
print("Tracing streamlines...")                                              # Progress message
all_paths = [trace_streamlines(u,v,w,y_flat,z_flat) for (u,v,w) in snapshots]  # Trace streamlines for every saved snapshot

Tracing streamlines...


In [12]:
#Sphere surface mesh
Nu_s, Nv_s = 20, 30                                                          # Mesh resolution: 20 polar divisions, 30 azimuthal divisions
theta = np.linspace(0,np.pi,Nu_s); phi = np.linspace(0,2*np.pi,Nv_s)         # Polar angle 0..pi, azimuthal angle 0..2pi
THETA,PHI = np.meshgrid(theta,phi,indexing='ij')                             # 2D grids of angles
sx = cx + R*np.sin(THETA)*np.cos(PHI)                                        # Sphere x-coordinates (spherical -> Cartesian)
sy = cy + R*np.sin(THETA)*np.sin(PHI)                                        # Sphere y-coordinates
sz = cz + R*np.cos(THETA)                                                    # Sphere z-coordinates

In [13]:
#Rendering the animation
fig = plt.figure(figsize=(10,7))                                             # Create a 10x7 inch figure
ax = fig.add_subplot(111, projection='3d')                                   # Add a single 3D axes
def update(frame):                                                           # Called once per animation frame
    ax.clear()                                                               # Wipe the previous frame
    ax.plot_wireframe(sx, sy, sz, rstride=1, cstride=1, linewidth=0.4, color='gray', alpha=0.6)  # Draw the sphere as a gray wireframe
    pts_all = all_paths[frame]                                               # Streamlines for this frame: (n_steps+1, n_seeds, 3)
    for i in range(pts_all.shape[1]):                                        # Loop over each streamline
        pts = pts_all[:,i,:]                                                 # Points of streamline i: (n_steps+1, 3)
        seg = np.stack([pts[:-1],pts[1:]],axis=1)                            # Pair consecutive points into line segments
        lc = Line3DCollection(seg, colors="#00d5ff", linewidth=2.5, alpha=0.9)  # Build a cyan line collection from the segments
        ax.add_collection3d(lc)                                              # Add it to the 3D plot
    ax.set_xlim(0,Nx); ax.set_ylim(0,Ny); ax.set_zlim(0,Nz)                  # Fix axis limits so the view doesn't jump between frames
    ax.set_xlabel("x"); ax.set_ylabel("y"); ax.set_zlabel("z")               # Axis labels
    ax.set_title(f"Re={int(Re)}  --  frame {frame}/{len(snapshots)}  (longer, slower animation)", fontsize=11)  # Title with Re and frame number
    ax.view_init(elev=18, azim=-65)                                          # Fixed camera angle (elevation 18 deg, azimuth -65 deg)
print("Rendering...")                                                        # Progress message
ani = FuncAnimation(fig, update, frames=len(snapshots), interval=1000/FPS)   # Build the animation: one frame per snapshot
ani.save("sphere_long.gif", writer=PillowWriter(fps=FPS))                    # Save as a GIF at the chosen FPS
plt.close(fig)                                                               # Close the figure to free memory
print("saved sphere_long.gif")  

Rendering...
saved sphere_long.gif
